In [5]:
# import öncesinde terminalde çalıştırılan kodlar:
# env simocuk olarak seçtik.
# conda install -n simocuk pip --force-reinstall -y ,, "simocuk ortamındaki pip'i internetten sıfır ve sağlam haliyle indirip üzerine yaz"
# conda activate simocuk
# /Users/simonxji/anaconda3/envs/simocuk/bin/python -m pip install joblib pandas seaborn matplotlib scikit-learn scikit-uplift catboost lightgbm xgboost

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
import missingno as msno
from datetime import date
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler


from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)

In [6]:
def load_application_train():
    data = pd.read_csv("/Users/simonxji/Desktop/miuul dsb final project/causal-crm-uplift-modeling/hillstrom_kampanya_verisi.csv")
    return data

df = load_application_train()
df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,target,treatment
0,10,2) $100 - $200,142.440,1,0,Surburban,0,Phone,0,Womens E-Mail
1,6,3) $200 - $350,329.080,1,1,Rural,1,Web,0,No E-Mail
2,7,2) $100 - $200,180.650,0,1,Surburban,1,Web,0,Womens E-Mail
3,9,5) $500 - $750,675.830,1,0,Rural,1,Web,0,Mens E-Mail
4,2,1) $0 - $100,45.340,1,0,Urban,0,Web,0,Womens E-Mail


In [9]:
df_train, df_test = train_test_split(
    df, 
    test_size=0.30, 
    random_state=17, 
    stratify=df['target'] # veri dengesiz olduğu için gerekli! dengeli olarak dağıtır. alışveriş yapan/toplam oranı aynıdır.
)

df_train.to_csv("train_data.csv", index=False)
df_test.to_csv("test_data.csv", index=False)

print(f"Eğitim Seti: {df_train.shape}")
print(f"Test Seti: {df_test.shape}")

Eğitim Seti: (44800, 10)
Test Seti: (19200, 10)


In [ ]:
df.isnull().values.any()
df.head() 
# RFM (recency, frequency, monetary (history olarak adlandırılmış))
# history_segment, history'nin gruplanmış hali. sadece birini kullanmak yeterli. ikisi beraber yüksek korelasyon.
# sadece kadın veya sadece erkek ürünü alan kişiler de var. her bir index bir kişiyi temsil eder.
# newbie yeni üye olup olmadığı.
# channel, müşterinin geçmişte alışveriş yaparken hangi kanalı kullandığı
# treatment, müşteriye kadın/erkek emaili mi atıldı ya da hiç email atılmadı mı?
# target, e-posta atıldıktan sonraki 2 hafta içinde bu müşteri siteden alışveriş yaptı mı? tahmin etmeye çalıştığımız değer

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,target,treatment
0,10,2) $100 - $200,142.440,1,0,Surburban,0,Phone,0,Womens E-Mail
1,6,3) $200 - $350,329.080,1,1,Rural,1,Web,0,No E-Mail
2,7,2) $100 - $200,180.650,0,1,Surburban,1,Web,0,Womens E-Mail
3,9,5) $500 - $750,675.830,1,0,Rural,1,Web,0,Mens E-Mail
4,2,1) $0 - $100,45.340,1,0,Urban,0,Web,0,Womens E-Mail
